In [3]:
import random
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# CONFIG
# =========================
BASE_URL = "https://rebaltica.lv/petijumi/recheck/"
SOURCE_NAME = "Re:Baltica (Re:Check)"
DEFAULT_LABEL = 1

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120 Safari/537.36"
    ),
    "Accept-Language": "lv-LV,lv;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Connection": "keep-alive",
}

# Scraping speed / politeness
SLEEP_BETWEEN_PAGES_MIN = 1.5
SLEEP_BETWEEN_PAGES_MAX = 2.5

SLEEP_BETWEEN_ARTICLES_MIN = 1.5
SLEEP_BETWEEN_ARTICLES_MAX = 3.0

LONG_BREAK_EVERY = 50
LONG_BREAK_SECONDS = 20

SAVE_EVERY = 25
REQUEST_TIMEOUT = 30

# TF-IDF near-duplicate filtering
SIM_THRESHOLD = 0.92
MIN_TEXT_LEN = 200

sections = [
    "politiku-izteikumi",
    "velesanas",
    "eiropas-savieniba",
    "krievijas-iebrukums-ukraina",
    "klimats-un-energetika",
    "lgbtq",
    "migracija",
    "veseliba",
    "sazverestibas-teorijas",
    "cits",
]

# =========================
# SESSION / HTTP
# =========================
def create_session():
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=2,
        status_forcelist=[403, 429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )

    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update(HEADERS)

    return session

session = create_session()

def sleep_range(a, b):
    time.sleep(random.uniform(a, b))

def safe_get(url, timeout=REQUEST_TIMEOUT, max_manual_retries=4):
    """
    GET ar timeout, session, retry un papildu manual backoff.
    """
    for attempt in range(1, max_manual_retries + 1):
        try:
            response = session.get(url, timeout=timeout)


            if response.status_code in (403, 429):
                wait_time = attempt * 10 + random.uniform(1, 3)
                print(f"  Server returned {response.status_code} for {url}")
                print(f"  Waiting {wait_time:.1f}s before retry...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()
            return response

        except requests.exceptions.RequestException as e:
            wait_time = attempt * 5 + random.uniform(1, 2)
            print(f"  Request failed (attempt {attempt}/{max_manual_retries}) for {url}")
            print(f"  Error: {type(e).__name__}: {e}")
            print(f"  Retrying in {wait_time:.1f}s...")
            time.sleep(wait_time)

    raise RuntimeError(f"Failed after retries: {url}")

# =========================
# HELPERS
# =========================
def get_total_pages(section_url):
    soup = BeautifulSoup(safe_get(section_url).content, "html.parser")
    last_page = 1

    for a_tag in soup.select("a.page-numbers"):
        try:
            num = int(a_tag.get_text(strip=True))
            last_page = max(last_page, num)
        except ValueError:
            continue

    return last_page

def get_article_links(page_url):
    soup = BeautifulSoup(safe_get(page_url).content, "html.parser")
    links = []

    for h2_tag in soup.find_all("h2"):
        a_tag = h2_tag.find("a", href=True)
        if not a_tag:
            continue

        href = a_tag["href"]
        if "/20" in href:
            full_url = href if href.startswith("http") else urljoin("https://rebaltica.lv", href)
            links.append(full_url)

    return links

def scrape_article(url):
    soup = BeautifulSoup(safe_get(url).content, "html.parser")

    title_tag = soup.find("h1") or soup.find("h2")
    title = title_tag.get_text(strip=True) if title_tag else ""

    date_tag = soup.find("time", class_="ct-meta-element-date")
    publication_date = ""
    if date_tag and date_tag.has_attr("datetime"):
        publication_date = date_tag["datetime"][:10]

    article_container = soup.find("article")
    paragraphs = article_container.find_all("p") if article_container else soup.find_all("p")
    full_text = " ".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
        if p.get_text(strip=True)
    )

    return title, full_text, publication_date

def save_progress(records, filename="rebaltica_progress.csv"):
    df = pd.DataFrame(records)
    df.to_csv(filename, index=False, encoding="utf-8")
    print(f"  Progress saved: {filename} ({len(df)} records)")

def tfidf_near_duplicate_filter(df, text_col="full_text", threshold=0.92, min_text_len=200):
    df = df.copy().reset_index(drop=True)

    mask_long = df[text_col].fillna("").str.len() >= min_text_len
    idx_long = df.index[mask_long].tolist()

    if len(idx_long) < 2:
        audit_df = pd.DataFrame(columns=["kept_index", "removed_index", "similarity"])
        return df, audit_df

    texts = df.loc[idx_long, text_col].fillna("").tolist()

    vectorizer = TfidfVectorizer(
        lowercase=True,
        max_features=50000,
        ngram_range=(1, 2),
    )
    X = vectorizer.fit_transform(texts)
    sim = cosine_similarity(X, X)

    to_remove_local = set()
    audit_rows = []

    for i in range(sim.shape[0]):
        if i in to_remove_local:
            continue

        for j in range(i + 1, sim.shape[0]):
            if j in to_remove_local:
                continue

            if sim[i, j] >= threshold:
                kept_idx = idx_long[i]
                removed_idx = idx_long[j]
                to_remove_local.add(j)

                audit_rows.append({
                    "kept_index": kept_idx,
                    "removed_index": removed_idx,
                    "similarity": float(sim[i, j]),
                })

    remove_df_indices = sorted({idx_long[j] for j in to_remove_local})
    df_dedup = df.drop(index=remove_df_indices).reset_index(drop=True)

    if audit_rows:
        audit_df = pd.DataFrame(audit_rows).sort_values(by="similarity", ascending=False)
    else:
        audit_df = pd.DataFrame(columns=["kept_index", "removed_index", "similarity"])

    return df_dedup, audit_df

# =========================
# MAIN: COLLECT LINKS
# =========================
print("Starting Re:Check article collection...\n")

url_to_topic = {}
all_links = []

for section in sections:
    section_url = urljoin(BASE_URL, section + "/")
    print(f"Processing section/topic: {section}")

    try:
        total_pages = get_total_pages(section_url)
        print(f"  Total pages detected: {total_pages}")
    except Exception as e:
        print(f"  Failed to read section: {section_url} | {type(e).__name__}: {e}")
        continue

    for page in range(1, total_pages + 1):
        page_url = section_url if page == 1 else urljoin(section_url, f"page/{page}/")

        try:
            links = get_article_links(page_url)
            all_links.extend(links)
            for u in links:
                url_to_topic[u] = section

            print(f"  Page {page}/{total_pages}: {len(links)} links")
        except Exception as e:
            print(f"  Error reading {page_url}: {type(e).__name__}: {e}")

        sleep_range(SLEEP_BETWEEN_PAGES_MIN, SLEEP_BETWEEN_PAGES_MAX)

print(f"\nTotal collected links (incl. duplicates): {len(all_links)}")
unique_links = sorted(set(all_links))
print(f"Unique article links after URL deduplication: {len(unique_links)}")

# =========================
# MAIN: SCRAPE ARTICLES
# =========================
records = []
failed_urls = []
N = len(unique_links)

for i, url in enumerate(unique_links, start=1):
    print(f"Scraping {i}/{N}: {url}")

    try:
        title, full_text, publication_date = scrape_article(url)
        topic = url_to_topic.get(url, "")

        records.append({
            "title": title,
            "full_text": full_text,
            "source": SOURCE_NAME,
            "publication_date": publication_date,
            "topic": topic,
            "class_label": DEFAULT_LABEL,
            "url": url,
        })

    except Exception as e:
        print(f"  Error scraping {url}: {type(e).__name__}: {e}")
        failed_urls.append(url)


    if i % SAVE_EVERY == 0:
        save_progress(records, "rebaltica_progress.csv")


    if i % LONG_BREAK_EVERY == 0:
        print(f"  Taking a long break for {LONG_BREAK_SECONDS}s...")
        time.sleep(LONG_BREAK_SECONDS)

    sleep_range(SLEEP_BETWEEN_ARTICLES_MIN, SLEEP_BETWEEN_ARTICLES_MAX)

# Save raw
df_raw = pd.DataFrame(records)
print(f"\nScraped records (raw): {len(df_raw)}")

raw_file = "rebaltica_recheck_raw.csv"
df_raw.to_csv(raw_file, index=False, encoding="utf-8")
print(f"Saved raw CSV: {raw_file}")

# Save failed URL, if any
if failed_urls:
    pd.DataFrame({"url": failed_urls}).to_csv("rebaltica_failed_urls.csv", index=False, encoding="utf-8")
    print(f"Saved failed URLs: rebaltica_failed_urls.csv ({len(failed_urls)} failed)")

# =========================
# TF-IDF DEDUPLICATION
# =========================
print("\nRunning TF-IDF near-duplicate filtering...")

df_dedup, audit = tfidf_near_duplicate_filter(
    df_raw,
    text_col="full_text",
    threshold=SIM_THRESHOLD,
    min_text_len=MIN_TEXT_LEN,
)

removed = len(df_raw) - len(df_dedup)
print(f"Near-duplicates removed (cosine >= {SIM_THRESHOLD}): {removed}")
print(f"Remaining records after TF-IDF dedup: {len(df_dedup)}")

dedup_file = "rebaltica_recheck_clean1.csv"
audit_file = "rebaltica_recheck_dedup_audit.csv"

df_dedup.to_csv(dedup_file, index=False, encoding="utf-8")
audit.to_csv(audit_file, index=False, encoding="utf-8")

print(f"Saved deduplicated CSV: {dedup_file}")
print(f"Saved audit CSV: {audit_file}")
print("\nDone.")

Starting Re:Check article collection...

Processing section/topic: politiku-izteikumi
  Total pages detected: 14
  Page 1/14: 6 links
  Page 2/14: 6 links
  Page 3/14: 6 links
  Page 4/14: 6 links
  Page 5/14: 6 links
  Page 6/14: 6 links
  Page 7/14: 6 links
  Page 8/14: 6 links
  Page 9/14: 6 links
  Page 10/14: 6 links
  Page 11/14: 6 links
  Page 12/14: 6 links
  Page 13/14: 6 links
  Page 14/14: 1 links
Processing section/topic: velesanas
  Total pages detected: 7
  Page 1/7: 6 links
  Page 2/7: 6 links
  Page 3/7: 6 links
  Page 4/7: 6 links
  Page 5/7: 6 links
  Page 6/7: 6 links
  Page 7/7: 3 links
Processing section/topic: eiropas-savieniba
  Total pages detected: 6
  Page 1/6: 6 links
  Page 2/6: 6 links
  Page 3/6: 6 links
  Page 4/6: 6 links
  Page 5/6: 6 links
  Page 6/6: 2 links
Processing section/topic: krievijas-iebrukums-ukraina
  Total pages detected: 5
  Page 1/5: 6 links
  Page 2/5: 6 links
  Page 3/5: 6 links
  Page 4/5: 6 links
  Page 5/5: 5 links
Processing secti

if blocked by source, use rebaltica_recheck_raw.csv form folder saved previously

1. Raw data inspection

In [4]:
import pandas as pd
import re
from collections import Counter

FILE = "rebaltica_recheck_raw.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# --------------------------------------------------
# 1) BASIC INSPECTION
# --------------------------------------------------
print("\nFIRST 5 TITLES:")
for i, title in enumerate(df["title"].fillna("").astype(str).head(5), start=1):
    print(f"{i}. {title}")

print("\nLAST 5 TITLES:")
for i, title in enumerate(df["title"].fillna("").astype(str).tail(5), start=1):
    print(f"{i}. {title}")

print("\nTEXT LENGTH SUMMARY:")
lengths = df["full_text"].fillna("").astype(str).str.len()
print("Min:", int(lengths.min()))
print("Median:", int(lengths.median()))
print("Max:", int(lengths.max()))

# --------------------------------------------------
# 2) TOP REPEATED SENTENCES
# --------------------------------------------------
print("\nTOP 40 REPEATED SENTENCES:")

all_sentences = []

for text in df["full_text"].fillna("").astype(str):
    sentences = re.split(r'(?<=[\.\!\?])\s+', text)

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) >= 40:
            all_sentences.append(sentence)

sentence_counts = Counter(all_sentences)

for sentence, count in sentence_counts.most_common(40):
    print(f"{count:>4}  {sentence[:250]}")

# --------------------------------------------------
# 3) TOKENIZATION
# --------------------------------------------------
def tokenize(text: str):
    return re.findall(r"[A-Za-zĀ-Žā-ž0-9\-:]+", text.lower())

all_words = []
all_bigrams = []
all_trigrams = []

for text in df["full_text"].fillna("").astype(str):
    tokens = tokenize(text)
    tokens = [t for t in tokens if len(t) >= 3]

    all_words.extend(tokens)
    all_bigrams.extend(zip(tokens, tokens[1:]))
    all_trigrams.extend(zip(tokens, tokens[1:], tokens[2:]))

# --------------------------------------------------
# 4) TOP WORDS
# --------------------------------------------------
print("\nTOP 50 WORDS:")
for word, count in Counter(all_words).most_common(50):
    print(f"{count:>4}  {word}")

# --------------------------------------------------
# 5) TOP BIGRAMS
# --------------------------------------------------
print("\nTOP 40 BIGRAMS:")
for bg, count in Counter(all_bigrams).most_common(40):
    print(f"{count:>4}  {' '.join(bg)}")

# --------------------------------------------------
# 6) TOP TRIGRAMS
# --------------------------------------------------
print("\nTOP 40 TRIGRAMS:")
for tg, count in Counter(all_trigrams).most_common(40):
    print(f"{count:>4}  {' '.join(tg)}")

# --------------------------------------------------
# 7) SOURCE / BRANDING CONTEXTS
# --------------------------------------------------
print("\nTOP CONTEXTS CONTAINING SOURCE WORDS:")

SOURCE_PATTERNS = {
    "re:check": r"\bre:?check\b",
    "re:baltica": r"\bre:?baltica\b",
    "rebaltica.com": r"\brebaltica\.com\b",
    "recheck@rebaltica.com": r"\brecheck@rebaltica\.com\b",
}

for label, pattern in SOURCE_PATTERNS.items():
    contexts = []

    for text in df["full_text"].fillna("").astype(str):
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            start = max(match.start() - 50, 0)
            end = min(match.end() + 120, len(text))
            context = text[start:end].strip()
            contexts.append(context)

    if contexts:
        print(f"\n--- {label.upper()} ---")
        for context, count in Counter(contexts).most_common(20):
            print(f"{count:>4}  {context}")

# --------------------------------------------------
# 8) FACT-CHECK CUE FAMILY COUNTS
# --------------------------------------------------
print("\nFACT-CHECK CUE FAMILY COUNTS (TEXT):")

CUE_FAMILIES = {
    "faktu pārbaud*": r"\bfaktu\s+pārbaud\w*\b",
    "maldin*": r"\bmaldin\w*\b",
    "nepaties*": r"\bnepaties\w*\b",
    "paties*": r"\bpaties\w*\b",
    "apgalvoj*": r"\bapgalvoj\w*\b",
    "secin*": r"\bsecin\w*\b",
    "kontekst*": r"\bkontekst\w*\b",
    "dezinform*": r"\bdezinform\w*\b",
    "propagand*": r"\bpropagand\w*\b",
    "nav taisnība": r"\bnav\s+taisn(?:ība|i)\b",
    "tuvu patiesībai": r"\btuvu\s+patiesībai\b",
    "puspaties*": r"\bpuspaties\w*\b",
}

full_text_series = df["full_text"].fillna("").astype(str)

for label, pattern in CUE_FAMILIES.items():
    count = full_text_series.str.contains(pattern, case=False, regex=True).sum()
    print(f"{count:>4}  {label}")

print("\nFACT-CHECK CUE FAMILY COUNTS (TITLES):")

title_series = df["title"].fillna("").astype(str)

for label, pattern in CUE_FAMILIES.items():
    count = title_series.str.contains(pattern, case=False, regex=True).sum()
    print(f"{count:>4}  {label}")

# --------------------------------------------------
# 9) TITLE PREFIXES BEFORE COLON
# --------------------------------------------------
print("\nTOP TITLE PREFIXES BEFORE COLON:")

prefixes = []

for title in title_series:
    match = re.match(r"^\s*([^:]{1,60}):", title)
    if match:
        prefixes.append(match.group(1).strip())

for prefix, count in Counter(prefixes).most_common(30):
    print(f"{count:>4}  {prefix}:")

# --------------------------------------------------
# 10) EXACT KEYWORD CHECKS
# --------------------------------------------------
print("\nEXACT KEYWORD CHECKS:")

keywords = [
    "re:check",
    "re:baltica",
    "rebaltica",
    "recheck@rebaltica.com",
    "nav taisnība",
    "apgalvojums",
    "secinājums",
    "trūkst konteksta",
    "patiesība",
    "tuvu patiesībai",
    "puspatiesība",
    "viedoklis",
]

for k in keywords:
    text_count = full_text_series.str.contains(k, case=False, regex=False).sum()
    title_count = title_series.str.contains(k, case=False, regex=False).sum()
    print(f"{k:>24} | text: {int(text_count):>4} | title: {int(title_count):>4}")

ROWS: 441
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url']

FIRST 5 TITLES:
1. Neuzticība norauj vienotu gāzes projektu
2. Nomuļļātais sašķidrinātās gāzes projekts
3. “Jauna Černobiļa Eiropā” – kā Krievija biedē ar radiāciju no Ukrainas?
4. Karstie meli – kā par klimata pārmaiņām maldina Baltijas valstīs?
5. Kas un kā Baltijas valstīs ar meliem kurina naidu pret LGBT+ kopienu?

LAST 5 TITLES:
1. Vai bērnu pabalstiem bijis lielākais finansējuma pieaugums desmitgadē?
2. Vai komunikācija par Rail Baltica bijusi “skaidra un atklāta”?
3. Vai Latvijā jau pielāgotas 500 patvertnes?
4. Vai nodokļos aiziet 50–60 % algas?
5. “Viss sagrauts” jeb kā kārtējais pārbēdzējs melo par dzīvi Latvijā

TEXT LENGTH SUMMARY:
Min: 495
Median: 5138
Max: 69674

TOP 40 REPEATED SENTENCES:
 581  Vērtējumu piešķir vismaz divi redaktori vienojoties.
 511  Pārbaudīto faktu vērtējumam izmantojam sešas iespējamās atzīmes un krāsas: patiesība (izteikums ir precīzs un faktoloģ

Step 1 inspection was used to separate source-identifying and boilerplate expressions from verdict-signalling lexical cues. Repeated references to Re:Check, Re:Baltica, contact details, funding messages, methodological descriptions, and donation prompts were assigned to source cleaning. In contrast, judgement-related expressions such as nav taisnība, apgalvoj-, secin-, trūkst konteksta, paties-, puspaties-, and nepaties- were assigned to the masking stage, because they directly signalled the fact-check verdict and could leak class-specific information.

2. Cleaning

In [5]:
%%writefile clean_rebaltica_recheck.py
from __future__ import annotations

import re
import html
import pandas as pd

IN_FILE = "rebaltica_recheck_raw.csv"
OUT_FILE = "rebaltica_recheck_clean.csv"

MIN_TEXT_LEN = 300

# --------------------------------------------------
# Title cleanup
# --------------------------------------------------
TITLE_PREFIX_PATTERNS = [
    r'^\s*Labots\s*:\s*',
    r'^\s*Skaidrojam\s*:\s*',
]
TITLE_TRAILING_COUNT_PATTERN = re.compile(r"\s*\(\d+\)\s*$")

# --------------------------------------------------
# Boilerplate / methodology / donation / contact / project text
# --------------------------------------------------
REMOVE_PATTERNS = [
    r'Vērtējumu\s+piešķir\s+vismaz\s+divi\s+redaktori\s+vienojoties\.?',
    r'Pārbaudīto\s+faktu\s+vērtējumam\s+izmantojam\s+sešas\s+iespējamās\s+atzīmes\s+un\s+krāsas:.*',
    r'Ja\s+arī\s+jūs\s+redzat\s+apšaubāmu\s+apgalvojumu,\s+sūtiet\s+to\s+mums\s+uz\s+recheck@rebaltica\.com.*',
    r'Ja\s+redzi\s+apšaubāmu\s+apgalvojumu,\s+sūti\s+to\s+uz\s+recheck@rebaltica\.com.*',
    r'Par\s+projektu:\s+Re:Check\s+ir\s+Baltijas\s+pētnieciskās\s+žurnālistikas\s+centra\s+Re:Baltica\s+paspārnē\s+strādājoša\s+faktu\s+pārbaudes\s+un\s+sociālo\s+tīklu\s+pētniecības\s+virtuāla\s+laboratorija\.?',
    r'Par\s+Re:Check\s+sadarbību\s+ar\s+Facebook\s+lasiet\s+šeit\.?',
    r'Par\s+mūsu\s+sadarbību\s+ar\s+META\s+lasi\s+šeit\.?',
    r'Visus\s+Re:Check\s+rakstus\s+lasi\s+ŠEIT!?\.?',
    r'Visus\s+Re:Check\s+rakstus\s+lasi\s+šeit\.?',
    r'Šis\s+raksts\s+ir\s+daļa\s+no\s+Re:Check\s+darba,.*',
    r'Konts:\s*LV38RIKO0001060112712\s*Tagad\s+ziedo\s+arī\s+ar\s+Mobilly!?\.?',
    r'Lai\s+to\s+izdarītu,\s+lietotnes\s+ziedojumu\s+sadaļā\s+atrodiet\s+mūsu\s+zīmolu\s+un\s+sekojiet\s+tālākajām\s+norādēm\.?',
    r'NEATKARĪGAI\s+ŽURNĀLISTIKAI\s+VAJAG\s+NEATKARĪGU\s+FINANSĒJUMU\s+Ja\s+Jums\s+patīk\s+Re:Baltica\s+darbs,\s+atbalstiet\s+mūs\s*!?',
    r'Neatkarīgai\s+žurnālistikai\s+vajag\s+neatkarīgu\s+finansējumu\s+Pētījumi\s+bieži\s+top\s+vairākus\s+mēnešus\.?',
    r'Ja\s+Tev\s+ir\s+svarīgi,\s+lai\s+tie\s+būtu\s+un\s+tos\s+bez\s+maksas\s+varētu\s+lasīt\s+[–-]\s+atbalsti\s+mūsu\s+darbu!?\.?',
    r'Plašāk\s+par\s+vērtējuma\s+skalu\s+un\s+mūsu\s+darba\s+metodēm\s+lasi\s+šeit\s*\.?',
]

# Cut from first tail marker to end
TAIL_MARKERS = [
    r'Vērtējumu\s+piešķir\s+vismaz\s+divi\s+redaktori\s+vienojoties',
    r'Pārbaudīto\s+faktu\s+vērtējumam\s+izmantojam\s+sešas\s+iespējamās\s+atzīmes',
    r'Ja\s+arī\s+jūs\s+redzat\s+apšaubāmu\s+apgalvojumu',
    r'Ja\s+redzi\s+apšaubāmu\s+apgalvojumu',
    r'Par\s+projektu:\s+Re:Check\s+ir',
    r'Par\s+Re:Check\s+sadarbību\s+ar\s+Facebook',
    r'Par\s+mūsu\s+sadarbību\s+ar\s+META',
    r'Visus\s+Re:Check\s+rakstus\s+lasi',
    r'Šis\s+raksts\s+ir\s+daļa\s+no\s+Re:Check\s+darba',
    r'Konts:\s*LV38RIKO0001060112712',
    r'NEATKARĪGAI\s+ŽURNĀLISTIKAI\s+VAJAG\s+NEATKARĪGU\s+FINANSĒJUMU',
    r'Neatkarīgai\s+žurnālistikai\s+vajag\s+neatkarīgu\s+finansējumu',
    r'Plašāk\s+par\s+vērtējuma\s+skalu\s+un\s+mūsu\s+darba\s+metodēm',
]

# --------------------------------------------------
# Source / branding
# --------------------------------------------------
SOURCE_PATTERNS = [
    r'\bre:?check\b',
    r'\bre:?baltica\b',
    r'\brebaltica\b',
    r'\brecheck@rebaltica\.com\b',
    r'\bMETA\b',
    r'\bFacebook\b',
]

COMPILED_REMOVE_PATTERNS = [
    re.compile(pat, flags=re.IGNORECASE | re.UNICODE | re.DOTALL)
    for pat in REMOVE_PATTERNS + SOURCE_PATTERNS
]


def normalize_text(text: object) -> str:
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)

    text = re.sub(r"<[^>]+>", " ", text)

    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u00a0", " ").replace("\u200b", "")

    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)

    return text.strip()


def cut_from_first_marker(text: str, markers: list[str]) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    positions = []

    for pat in markers:
        m = re.search(pat, text, flags=re.IGNORECASE | re.UNICODE | re.DOTALL)
        if m:
            positions.append(m.start())

    if not positions:
        return text

    return text[:min(positions)].strip()


def clean_title(title: object) -> str:
    title = normalize_text(title)

    for pat in TITLE_PREFIX_PATTERNS:
        title = re.sub(pat, "", title, flags=re.IGNORECASE | re.UNICODE)

    title = TITLE_TRAILING_COUNT_PATTERN.sub("", title)
    title = re.sub(r"\s+", " ", title).strip()

    return title


def clean_full_text(text: object) -> str:
    text = normalize_text(text)

    # First cut long tails
    text = cut_from_first_marker(text, TAIL_MARKERS)

    # Then remove inline boilerplate / source branding
    for pattern in COMPILED_REMOVE_PATTERNS:
        text = pattern.sub(" ", text)

    # Cleanup leftovers
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?]){2,}", r"\1", text)
    text = re.sub(r"\(\s*\)", " ", text)
    text = re.sub(r"\[\s*\]", " ", text)
    text = re.sub(r"\s*-\s*-\s*", " - ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^[,.;:\-\s]+", "", text)

    return text.strip()


# --------------------------------------------------
# Load
# --------------------------------------------------
df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)
columns_before = df.columns.tolist()

# --------------------------------------------------
# Create cleaned columns
# --------------------------------------------------
df["title_clean"] = df["title"].apply(clean_title)
df["full_text_clean"] = df["full_text"].apply(clean_full_text)
df["text_length_clean"] = df["full_text_clean"].fillna("").astype(str).str.len()

# --------------------------------------------------
# Remove empty / too short texts
# --------------------------------------------------
df = df[df["full_text_clean"].fillna("").astype(str).str.strip().str.len() >= MIN_TEXT_LEN].copy()

rows_after = len(df)
removed_rows = rows_before - rows_after

df = df.reset_index(drop=True)

# --------------------------------------------------
# Save
# --------------------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

# --------------------------------------------------
# Report
# --------------------------------------------------
columns_after = df.columns.tolist()
new_columns = [c for c in columns_after if c not in columns_before]

print("Saved:", OUT_FILE)
print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Removed rows:", removed_rows)

print("\nColumns before:")
print(columns_before)

print("\nColumns after:")
print(columns_after)

print("\nNew columns added:")
print(new_columns if new_columns else "None")

Writing clean_rebaltica_recheck.py


In [6]:
!python clean_rebaltica_recheck.py

Saved: rebaltica_recheck_clean.csv
Rows before: 441
Rows after: 441
Removed rows: 0

Columns before:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url']

Columns after:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'title_clean', 'full_text_clean', 'text_length_clean']

New columns added:
['title_clean', 'full_text_clean', 'text_length_clean']


3. Masking

In [7]:
%%writefile mask_rebaltica_recheck.py
from __future__ import annotations

import re
import pandas as pd

IN_FILE = "rebaltica_recheck_clean.csv"
OUT_FILE = "rebaltica_recheck_masked.csv"
AUDIT_FILE = "rebaltica_recheck_mask_audit.csv"

MASK_TOKEN = "[MASK]"

# --------------------------------------------------
# Title masking
# Re:Check titles usually do not carry much source prefix,
# so here we focus on verdict / judgement cues only.
# --------------------------------------------------
TITLE_MASK_PATTERNS = [
    r'^\s*Nav\s+taisn(?:ība|i)\s*[:\-–—]?\s*',
    r'^\s*Drīzāk\s+nav\s+taisn(?:ība|i)\s*[:\-–—]?\s*',
    r'^\s*Trūkst\s+kontekst\w*\s*[:\-–—]?\s*',
    r'^\s*Secin\w*\s*[:\-–—]?\s*',
    r'^\s*Apgalvoj\w*\s*[:\-–—]?\s*',
    r'^\s*Nepaties\w*\s*[:\-–—]?\s*',
    r'^\s*Maldin\w*\s*[:\-–—]?\s*',
    r'^\s*Tuvu\s+patiesībai\s*[:\-–—]?\s*',
    r'^\s*Puspaties\w*\s*[:\-–—]?\s*',
]

# --------------------------------------------------
# Text masking
# This source has very strong verdict formulas, so we mask:
# - verdict labels
# - judgement cue families
# - manually found remaining fragments
# --------------------------------------------------
TEXT_MASK_PATTERNS = [
    # manually found remaining fragments first
    r'\bneatbilst\s+paties(?:ībai|ibu|ība)\s*,?\s*tam\s+nav\s+pierādījumu\s*,?\s*izteikuma\s+autors\s+neapzināti\s+maldina\s+vai\s+melo\b',
    r'\btiek\s+salīdzinātas\s+nesalīdzināmas\s+lietas\s*,?\s*izteikums\s+ir\s+pretrunā\s+ar\s+paša\s+iepriekš\s+teikto\s+vai\s+darīto\s+vai\s+trūkst\s+būtiskas\s+papildu\s+informācijas\b',
    r'\btiek\s+salīdzinātas\s+nesalīdzināmas\s+lietas\s*,?\s*izteikums\s+ir\s+pretrunā\s+ar\s+paša\s+iepriekš\s+teikto\s+vai\s+darīto\s*,?\s*vai\s+trūkst\s+būtiskas\s+papildu\s+informācijas\b',
    r'\bdrīzāk\s+nav\s+taisn(?:ība|i)\s+apgalvojumā\s+kripata\s+paties(?:ības|iba|ība)\s*,?\s*taču\s+nav\s+ņemti\s+vērā\s+būtiski\s+fakti\s+un/?vai\s+konteksts\s*,?\s*līdz\s+ar\s+to\s+izteikums\s+ir\s+maldinošs\b',

    # fixed longer verdict formulas first
    r'\bnav\s+taisn(?:ība|i)\b',
    r'\bdrīzāk\s+nav\s+taisn(?:ība|i)\b',
    r'\btrūkst\s+kontekst\w*\b',
    r'\btuvu\s+patiesībai\b',
    r'\bpuspaties\w*\b',
    r'\bfaktu\s+pārbaud\w*\b',

    # cue families
    r'\bnepaties\w*\b',
    r'\bmaldin\w*\b',
    r'\bapgalvoj\w*\b',
    r'\bsecin\w*\b',
    r'\bkontekst\w*\b',
    r'\bpaties\w*\b',
    r'\bdezinform\w*\b',
    r'\bpropagand\w*\b',
]

COMPILED_TITLE_MASK_PATTERNS = [
    re.compile(p, flags=re.IGNORECASE | re.UNICODE)
    for p in TITLE_MASK_PATTERNS
]

COMPILED_TEXT_MASK_PATTERNS = [
    re.compile(p, flags=re.IGNORECASE | re.UNICODE)
    for p in TEXT_MASK_PATTERNS
]


def safe_text(text: object) -> str:
    if pd.isna(text):
        return ""
    return str(text).strip()


def cleanup_masked_text(text: str) -> str:
    text = re.sub(r'(\[MASK\]\s*){2,}', MASK_TOKEN + " ", text)
    text = re.sub(r'\[MASK\]\s*[,;:]+\s*\[MASK\]', MASK_TOKEN, text)
    text = re.sub(r'\[MASK\]\s*[-–—:]\s*\[MASK\]', MASK_TOKEN, text)
    text = re.sub(r'\(\s*\[MASK\]\s*\)', MASK_TOKEN, text)
    text = re.sub(r'\[\s*\]', ' ', text)
    text = re.sub(r'^\s*[,;:.\-–—]+\s*', "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def mask_title(text: object) -> str:
    text = safe_text(text)

    for pattern in COMPILED_TITLE_MASK_PATTERNS:
        text = pattern.sub(MASK_TOKEN + " ", text)

    for pattern in COMPILED_TEXT_MASK_PATTERNS:
        text = pattern.sub(MASK_TOKEN, text)

    text = cleanup_masked_text(text)
    text = re.sub(r"^\[MASK\]\s*[:\-–—]\s*", MASK_TOKEN + " ", text)

    return text.strip()


def mask_full_text(text: object) -> str:
    text = safe_text(text)

    for pattern in COMPILED_TEXT_MASK_PATTERNS:
        text = pattern.sub(MASK_TOKEN, text)

    text = cleanup_masked_text(text)
    return text.strip()


df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

# ----------------------------------------
# Apply masking
# ----------------------------------------
df["title_masked"] = df["title_clean"].apply(mask_title)
df["full_text_masked"] = df["full_text_clean"].apply(mask_full_text)

# ----------------------------------------
# Mask diagnostics
# ----------------------------------------
df["title_changed_by_masking"] = df["title_clean"] != df["title_masked"]
df["text_changed_by_masking"] = df["full_text_clean"] != df["full_text_masked"]

df["title_mask_count"] = df["title_masked"].fillna("").astype(str).str.count(r"\[MASK\]")
df["text_mask_count"] = df["full_text_masked"].fillna("").astype(str).str.count(r"\[MASK\]")

# ----------------------------------------
# Save main masked file
# ----------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

# ----------------------------------------
# Save audit file
# ----------------------------------------
audit_df = df[
    df["title_changed_by_masking"] | df["text_changed_by_masking"]
][[
    "title",
    "title_clean",
    "title_masked",
    "full_text",
    "full_text_clean",
    "full_text_masked",
    "title_mask_count",
    "text_mask_count",
]].copy()

audit_df.to_csv(AUDIT_FILE, index=False, encoding="utf-8")

# ----------------------------------------
# Report
# ----------------------------------------
print("Saved:", OUT_FILE)
print("Saved audit:", AUDIT_FILE)
print("Rows:", rows_before)

print("\nMask summary:")
print({
    "title_changed_rows": int(df["title_changed_by_masking"].sum()),
    "text_changed_rows": int(df["text_changed_by_masking"].sum()),
    "total_title_masks": int(df["title_mask_count"].sum()),
    "total_text_masks": int(df["text_mask_count"].sum()),
})

print("\nNew columns added:")
print([
    "title_masked",
    "full_text_masked",
    "title_changed_by_masking",
    "text_changed_by_masking",
    "title_mask_count",
    "text_mask_count",
])

Writing mask_rebaltica_recheck.py


In [8]:
!python mask_rebaltica_recheck.py

Saved: rebaltica_recheck_masked.csv
Saved audit: rebaltica_recheck_mask_audit.csv
Rows: 441

Mask summary:
{'title_changed_rows': 16, 'text_changed_rows': 434, 'total_title_masks': 18, 'total_text_masks': 2398}

New columns added:
['title_masked', 'full_text_masked', 'title_changed_by_masking', 'text_changed_by_masking', 'title_mask_count', 'text_mask_count']


4. Remove mask

In [9]:
%%writefile remove_mask_rebaltica_recheck.py
from __future__ import annotations

import re
import pandas as pd

IN_FILE = "rebaltica_recheck_masked.csv"
OUT_FILE = "rebaltica_recheck_nomask.csv"

MASK_PATTERN = r"\[MASK\]"


def remove_mask_tokens(text: object) -> str:
    if pd.isna(text):
        return ""

    text = str(text)

    # remove mask token
    text = re.sub(MASK_PATTERN, " ", text, flags=re.IGNORECASE)

    # remove awkward leading punctuation
    text = re.sub(r"^\s*[,;:.\-–—]+\s*", "", text)

    # clean punctuation leftovers after mask removal
    text = re.sub(r"\s*[-–—:]\s*(?=[,.;!?)]|$)", " ", text)
    text = re.sub(r"([.!?])\s*[,;:.\-–—]+\s*", r"\1 ", text)

    # remove empty brackets
    text = re.sub(r"\(\s*\)", " ", text)
    text = re.sub(r"\[\s*\]", " ", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

before_title_masks = df["title_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()
before_text_masks = df["full_text_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()

df["title_masked"] = df["title_masked"].apply(remove_mask_tokens)
df["full_text_masked"] = df["full_text_masked"].apply(remove_mask_tokens)

after_title_masks = df["title_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()
after_text_masks = df["full_text_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()

df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Saved:", OUT_FILE)
print("Rows:", rows_before)
print("\nMask removal summary:")
print({
    "title_masks_before": int(before_title_masks),
    "text_masks_before": int(before_text_masks),
    "title_masks_after": int(after_title_masks),
    "text_masks_after": int(after_text_masks),
})

Writing remove_mask_rebaltica_recheck.py


In [10]:
!python remove_mask_rebaltica_recheck.py

Saved: rebaltica_recheck_nomask.csv
Rows: 441

Mask removal summary:
{'title_masks_before': 18, 'text_masks_before': 2398, 'title_masks_after': 0, 'text_masks_after': 0}


5. source level drdublication

In [11]:
%%writefile dedup_rebaltica_recheck.py
from __future__ import annotations

import pandas as pd
import hashlib

IN_FILE = "rebaltica_recheck_nomask.csv"
OUT_FILE = "rebaltica_recheck_dedup.csv"


def normalize_text(text: str) -> str:
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()
    text = " ".join(text.split())

    return text


def text_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()


df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

# --------------------------------------------------
# URL duplicates
# --------------------------------------------------
if "url" in df.columns:
    df["is_duplicate_url"] = df.duplicated(subset=["url"], keep="first")
else:
    df["is_duplicate_url"] = False

# --------------------------------------------------
# TEXT duplicates
# --------------------------------------------------
df["text_norm"] = df["full_text_masked"].apply(normalize_text)
df["text_hash"] = df["text_norm"].apply(text_hash)

df["is_duplicate_text"] = df.duplicated(subset=["text_hash"], keep="first")

# --------------------------------------------------
# Report
# --------------------------------------------------
dup_url = int(df["is_duplicate_url"].sum())
dup_text = int(df["is_duplicate_text"].sum())

# --------------------------------------------------
# Drop helper columns
# --------------------------------------------------
df = df.drop(columns=["text_norm", "text_hash"])

# --------------------------------------------------
# Save
# --------------------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Saved:", OUT_FILE)

print("\nRows:", rows_before)

print("\nDuplicates summary:")
print({
    "duplicate_urls": dup_url,
    "duplicate_texts": dup_text
})

print("\nColumns added:")
print(["is_duplicate_url", "is_duplicate_text"])

Writing dedup_rebaltica_recheck.py


In [12]:
!python dedup_rebaltica_recheck.py

Saved: rebaltica_recheck_dedup.csv

Rows: 441

Duplicates summary:
{'duplicate_urls': 0, 'duplicate_texts': 0}

Columns added:
['is_duplicate_url', 'is_duplicate_text']


6. Source level audit

In [13]:
%%writefile audit_rebaltica_recheck.py
from __future__ import annotations

import pandas as pd
import re
from collections import Counter

FILE = "rebaltica_recheck_dedup.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# --------------------------------------------------
# 1) Text length summary
# --------------------------------------------------
print("\nTEXT LENGTH SUMMARY:")
lengths = df["full_text_masked"].fillna("").astype(str).str.len()
print("Min:", int(lengths.min()))
print("Median:", float(lengths.median()))
print("Max:", int(lengths.max()))

# --------------------------------------------------
# 2) Top repeated sentences
# --------------------------------------------------
print("\nTOP 40 REPEATED SENTENCES:")

all_sentences = []

for text in df["full_text_masked"].fillna("").astype(str):
    sentences = re.split(r'(?<=[\.\!\?])\s+', text)

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) >= 40:
            all_sentences.append(sentence)

sentence_counts = Counter(all_sentences)

for sentence, count in sentence_counts.most_common(40):
    print(f"{count:>4}  {sentence[:250]}")

# --------------------------------------------------
# 3) Exact keyword checks in text
# --------------------------------------------------
print("\nEXACT KEYWORD CHECKS (TEXT):")

TEXT_KEYWORDS = [
    "re:check",
    "re:baltica",
    "rebaltica",
    "recheck@rebaltica.com",
    "facebook",
    "meta",
    "nav taisnība",
    "drīzāk nav taisnība",
    "trūkst konteksta",
    "secinājums",
    "apgalvojums",
    "patiesība",
    "tuvu patiesībai",
    "puspatiesība",
    "maldina",
    "nepatiesa",
]

text_series = df["full_text_masked"].fillna("").astype(str)

for k in TEXT_KEYWORDS:
    count = text_series.str.contains(k, case=False, regex=False).sum()
    print(f"{k:>24} : {int(count)}")

# --------------------------------------------------
# 4) Exact keyword checks in titles
# --------------------------------------------------
print("\nEXACT KEYWORD CHECKS (TITLES):")

title_series = df["title_masked"].fillna("").astype(str)

for k in TEXT_KEYWORDS:
    count = title_series.str.contains(k, case=False, regex=False).sum()
    print(f"{k:>24} : {int(count)}")

# --------------------------------------------------
# 5) Contexts for remaining source words
# --------------------------------------------------
print("\nTOP CONTEXTS FOR REMAINING SOURCE WORDS:")

SOURCE_PATTERNS = {
    "re:check": r"\bre:?check\b",
    "re:baltica": r"\bre:?baltica\b",
    "rebaltica.com": r"\brebaltica\.com\b",
    "facebook": r"\bfacebook\b",
    "meta": r"\bmeta\b",
}

for label, pattern in SOURCE_PATTERNS.items():
    contexts = []

    for text in text_series:
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            start = max(match.start() - 50, 0)
            end = min(match.end() + 100, len(text))
            context = text[start:end].strip()
            contexts.append(context)

    if contexts:
        print(f"\n--- {label.upper()} ---")
        context_counts = Counter(contexts)
        for context, count in context_counts.most_common(20):
            print(f"{count:>4}  {context}")

# --------------------------------------------------
# 6) Duplicate summary
# --------------------------------------------------
if "is_duplicate_url" in df.columns or "is_duplicate_text" in df.columns:
    print("\nDUPLICATE SUMMARY:")
    if "is_duplicate_url" in df.columns:
        print("Duplicate URLs:", int(df["is_duplicate_url"].sum()))
    if "is_duplicate_text" in df.columns:
        print("Duplicate texts:", int(df["is_duplicate_text"].sum()))

# --------------------------------------------------
# 7) Masking summary
# --------------------------------------------------
if "title_mask_count" in df.columns and "text_mask_count" in df.columns:
    print("\nMASKING SUMMARY:")
    print("Rows with title masks:", int((df["title_mask_count"] > 0).sum()))
    print("Rows with text masks:", int((df["text_mask_count"] > 0).sum()))
    print("Total title masks:", int(df["title_mask_count"].sum()))
    print("Total text masks:", int(df["text_mask_count"].sum()))

# --------------------------------------------------
# 8) Empty checks
# --------------------------------------------------
print("\nEMPTY TEXTS:")
print(int((df["full_text_masked"].fillna("").astype(str).str.strip() == "").sum()))

print("\nEMPTY TITLES:")
print(int((df["title_masked"].fillna("").astype(str).str.strip() == "").sum()))

Writing audit_rebaltica_recheck.py


In [14]:
!python audit_rebaltica_recheck.py

ROWS: 441
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'title_clean', 'full_text_clean', 'text_length_clean', 'title_masked', 'full_text_masked', 'title_changed_by_masking', 'text_changed_by_masking', 'title_mask_count', 'text_mask_count', 'is_duplicate_url', 'is_duplicate_text']

TEXT LENGTH SUMMARY:
Min: 481
Median: 3230.0
Max: 67787

TOP 40 REPEATED SENTENCES:
   6  Cilvēki mēdz saskarties arī ar smagākām blaknēm, bet tās ir ļoti retas.
   6  Pēdējo 100 gadu laikā vairums šo infekciju ir iegrožotas vai pilnībā apkarotas, pateicoties vakcīnām.
   6  Dažkārt svītras pazūd gandrīz uzreiz, bet citreiz tās var palikt redzamas pat desmitiem minūšu; to ietekmē atmosfēras mitrums un temperatūra.
   5  Tas ir neredzams starojums, ko jūtam kā siltumu.
   5  Viņi , ka tā neizraisa veselības traucējumus.
   5  Apkopojot virkni pētījumu, Džonsa Hopkinsa Universitātes Sabiedrības veselības skolas zinātnieki ir , ka cilvēkiem, kas saņēmuši vismaz pir